# Task D — six-prior identification overlay, all priors on one device

Pre-registered: `taskc/DECISIONS.md` §14 (priors, 10⁶ solve draw, gate rule, ordering prediction μ < κ < ρ). **Every prior is retrained here, including `base` and `retrain2`**, so the between-prior band is not mixing devices — the earlier MPS runs are not reused.

Per prior: train (frozen recipe) → gate against **its own** simulator (G4 bands rebuilt from that simulator for misspecified priors, §14) → 10⁶ solve draw → duals C0–C3 → evaluation on draw C. A prior that fails its gate gets one retry with a different init seed; failing twice it drops from the overlay and the drop is reported. Writes to Drive after every prior, so an interrupted session resumes by re-running with `SKIP_DONE = True`.

A100 runtime ≈ 60–75 min for all six.

In [ ]:
PINNED_COMMIT    = "__PINNED__"          # the code this notebook was written and dry-run against
NOTEBOOK_VERSION = "overlay6-2026.09.22a"
EXPECT_TASKC     = "taskc-2026.09.22a"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned commit :", PINNED_COMMIT); print("checked out   :", HEAD); print("notebook      :", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), (
    f"checkout did not land on the pinned commit (HEAD={HEAD}, pinned={PINNED_COMMIT}) -- stop")
!git log --oneline -1

In [ ]:
import os, sys, json, time, math
import torch

# full fp32 -- set before any model code touches a tensor
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import asdict, replace
import taskc
from taskc.config import CFG, FROZEN, frozen
from taskc.priors import PRIORS
from taskc.data import build_training_set, make_loader, reference_paths
from taskc.ptheta import make_schedule, build_model, train_ptheta, save_checkpoint, sample_ptheta, save_draw, load_draw, draw_path
from taskc.gate import run_gate, print_report, summary_dict
from taskc.dual import solve_level, evaluate_on, baseline_on, TASKB, TASKB_EXOTICS_P, TASKB_EXOTICS_Q
from taskc.gate import sv_stats
from config import q_params, EXOTICS
assert taskc.__version__ == EXPECT_TASKC, f"stale notebook or checkout: taskc.__version__={taskc.__version__}, expected {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open("notebooks/taskd_02_overlay_colab.ipynb").read(), (
    "the notebook running here is not the one committed at the pinned commit -- re-open it from the pinned URL")

def close(a, b, rtol=1e-6, atol=1e-12):
    """Relative-tolerance float comparison; never ==."""
    return math.isclose(float(a), float(b), rel_tol=rtol, abs_tol=atol)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("tf32 matmul:", torch.backends.cuda.matmul.allow_tf32, "| tf32 cudnn:", torch.backends.cudnn.allow_tf32,
      "| matmul precision:", torch.get_float32_matmul_precision(), "| taskc:", taskc.__version__)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_option_pricing/artifacts_overlay"
else:
    DRIVE = "artifacts_overlay_local"
os.makedirs(DRIVE, exist_ok=True); print("writing to", DRIVE)

## Config — the six priors, one recipe

Every prior trains with `frozen(...)`: 450 epochs, cosine lr 1e-3 → 1e-5, EMA 0.999, batch 512 — byte-identical to the recipe the frozen P_θ used. (`CFG` keeps decay/EMA off for the ladder rungs; using it here would train the constant-lr model that fails the gate.)

In [ ]:
PRIOR_LIST = ["base", "retrain1", "retrain2", "mu25", "rho02", "kappa6"]
RETRY_SEED = {"base": 3, "retrain1": 4, "retrain2": 5, "mu25": 6, "rho02": 7, "kappa6": 8}   # one retry each (section 14)
SKIP_DONE  = True        # resume: skip priors whose sweep.json already exists in Drive
LEVELS = ["C0", "C1", "C2", "C3"]
q = q_params()
for t in PRIOR_LIST:
    p = PRIORS[t]; print(f"  {t:9s} drift={p.heston.drift} kappa={p.heston.kappa} rho={p.heston.rho} theta={p.heston.theta} init_seed={p.init_seed}  ({p.note})")
print(f"\nsolve draw {CFG.solve_draw.n:,} (seed {CFG.solve_draw.seed}); eval draw C {CFG.draws['C'].n:,}; gate ref seed {CFG.ref_seed}")
print(f"recipe: epochs {FROZEN.epochs}, lr {FROZEN.lr} -> {FROZEN.lr_min} cosine, EMA {FROZEN.ema_decay}, batch {FROZEN.batch_size}")

## Run every prior: train → gate (own simulator) → 10⁶ solve draw → duals C0–C3

In [ ]:
def run_prior(tag, init_seed=None, attempt=1):
    pr = PRIORS[tag]
    out = os.path.join(DRIVE, tag if attempt == 1 else f"{tag}_retry")
    os.makedirs(out, exist_ok=True)
    cfg = frozen(artifact_dir=out, heston=pr.heston, init_seed=pr.init_seed if init_seed is None else init_seed)
    # frozen() = 450 epochs, cosine lr 1e-3 -> 1e-5, EMA 0.999 -- identical to the frozen P_theta's recipe
    rec = dict(tag=tag, attempt=attempt, init_seed=cfg.init_seed, heston=asdict(cfg.heston), device=DEVICE, timings={})
    ts = build_training_set(cfg); sched = make_schedule(cfg, device=DEVICE)
    t0 = time.time(); model = build_model(cfg).to(DEVICE)
    model = train_ptheta(model, make_loader(ts.z, cfg.batch_size, seed=cfg.init_seed), sched, cfg, device=DEVICE)
    rec["timings"]["train_s"] = time.time() - t0
    # the sampled weights must be the EMA ones, not the raw last iterate
    assert hasattr(model, "raw_state_dict"), "decay+EMA loop did not run (check cfg.lr_decay / cfg.ema)"
    k0 = next(k for k, v in model.state_dict().items() if v.dtype.is_floating_point)
    ema_w, raw_w = model.state_dict()[k0].detach().cpu(), model.raw_state_dict[k0]
    assert not torch.allclose(ema_w, raw_w), "sampled weights equal the raw last iterate -- EMA was not applied"
    rec["ema_drift"] = float((ema_w - raw_w).abs().max())
    print(f"  EMA applied: max|EMA - raw| = {rec['ema_drift']:.3e}")
    save_checkpoint(os.path.join(out, cfg.ckpt_name), model, ts.std, cfg, extra=dict(tag=tag, attempt=attempt, device=DEVICE))
    t0 = time.time()
    draws = {}
    for nm in ("A", "B", "C"):
        d = cfg.draws[nm]; draws[nm] = sample_ptheta(model, sched, d.n, d.seed, cfg, DEVICE, verbose=False); save_draw(cfg, nm, draws[nm])
    rec["timings"]["draws_s"] = time.time() - t0
    ref = reference_paths(cfg)                      # the prior's OWN simulator
    gate = run_gate(draws["A"].z, ts.std, cfg, ref=ref)
    rec["gate"] = summary_dict(gate); rec["sv_ref"] = sv_stats(ref.returns(), cfg.dt)
    print(f"\n=== {tag} (attempt {attempt}, seed {cfg.init_seed}): gate {'PASSED' if gate.passed else 'FAILED'}")
    print_report(gate, columns=False)
    json.dump(rec, open(os.path.join(out, "prior.json"), "w"), indent=1, default=float)
    if not gate.passed:
        return rec, None, None, None, ts, sched, model, cfg
    t0 = time.time(); S6 = sample_ptheta(model, sched, cfg.solve_draw.n, cfg.solve_draw.seed, cfg, DEVICE, verbose=False); save_draw(cfg, "S6", S6)
    rec["timings"]["solve_draw_s"] = time.time() - t0
    pS6, pC = ts.std.to_paths(S6.z, "S6"), ts.std.to_paths(draws["C"].z, "C")
    sweep = dict(tag=tag, heston=asdict(cfg.heston), device=DEVICE, n_solve=int(S6.z.shape[0]), n_eval=int(draws["C"].z.shape[0]),
                 level_S6=float(S6.z.mean(0).sum()), level_C=float(draws["C"].z.mean(0).sum()), sv_C=sv_stats(pC.returns(), cfg.dt),
                 unweighted_C=dict(**{k: v for k, v in baseline_on(pC, q).items() if k in ("ho_van_rmse","ho_mart_rmse","mart_prof_max")},
                                   exotics={k: [float(a), float(b)] for k, (a, b) in baseline_on(pC, q)["exotics"].items()}), levels={})
    t0 = time.time()
    for lvl in LEVELS:
        tilt, r, cs = solve_level(lvl, pS6, q); e = evaluate_on(tilt, pC, q); e_in = evaluate_on(tilt, pS6, q)
        sweep["levels"][lvl] = dict(m=cs.m, screen_ok=bool(r.screen["all_ok"]), screen_margin=float(r.screen["margin"].min()),
            screen_bite=cs.names[int(np.argmin(r.screen["margin"]))], beta_raw_norm=float(np.linalg.norm(r.beta_raw)),
            ess_solve=float(r.ess_frac), ess_C=float(e["ess_frac"]), kl=float(r.kl), E_C_L=float(e["E_L"]),
            ho_van_in=float(e_in["ho_van_rmse"]), ho_van_C=float(e["ho_van_rmse"]), ho_mart_C=float(e["ho_mart_rmse"]),
            exotics_C={k: [float(a), float(b)] for k, (a, b) in e["exotics"].items()}, converged=bool(r.converged))
        print(f"  {lvl}: m={cs.m} margin {sweep['levels'][lvl]['screen_margin']:.3f} |b_raw| {sweep['levels'][lvl]['beta_raw_norm']:.2f} ESS_C {e['ess_frac']*100:.2f}% hoVan_in {e_in['ho_van_rmse']:.4f}")
    rec["timings"]["duals_s"] = time.time() - t0
    os.makedirs(os.path.join(out, "sweep"), exist_ok=True)
    json.dump(sweep, open(os.path.join(out, "sweep", "sweep.json"), "w"), indent=1, default=float)
    json.dump(rec, open(os.path.join(out, "prior.json"), "w"), indent=1, default=float)
    return rec, sweep, draws, S6, ts, sched, model, cfg

results = {}
for tag in PRIOR_LIST:
    done = os.path.join(DRIVE, tag, "sweep", "sweep.json")
    if SKIP_DONE and os.path.exists(done):
        results[tag] = json.load(open(done)); print(f"{tag}: already done, loaded"); continue
    t0 = time.time()
    rec, sweep, *_ = run_prior(tag)
    if sweep is None:                                   # one retry, section 14
        print(f"  {tag}: retrying once with init seed {RETRY_SEED[tag]}")
        rec, sweep, *_ = run_prior(tag, init_seed=RETRY_SEED[tag], attempt=2)
    if sweep is None:
        print(f"  *** {tag} FAILED its gate twice -> DROPPED from the overlay (reported, not omitted)")
    else:
        results[tag] = sweep
    print(f"  {tag} finished in {(time.time()-t0)/60:.1f} min")
    json.dump({k: v for k, v in results.items()}, open(os.path.join(DRIVE, "overlay.json"), "w"), indent=1, default=float)
print("\npriors swept:", list(results), "| dropped:", [t for t in PRIOR_LIST if t not in results])

## The identification figure and the C3 spread table

In [ ]:
LAB = {"base": "base", "retrain1": "retrain 1", "retrain2": "retrain 2", "mu25": "mu=0.25", "rho02": "rho=-0.2", "kappa6": "kappa=6"}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2)); x = np.arange(len(LEVELS) + 1)
for axi, k in zip(ax, EXOTICS):
    for tag, s in results.items():
        v = [s["unweighted_C"]["exotics"][k][0]] + [s["levels"][l]["exotics_C"][k][0] for l in LEVELS]
        e = [s["unweighted_C"]["exotics"][k][1]] + [s["levels"][l]["exotics_C"][k][1] for l in LEVELS]
        ls = "-" if tag == "base" else ("--" if tag.startswith("retrain") else "-.")
        axi.errorbar(x, v, yerr=[2*t for t in e], marker="o", ms=3, ls=ls, lw=2 if tag=="base" else 1.2, capsize=2, label=LAB[tag])
    axi.axhline(TASKB_EXOTICS_Q[k], c="k", ls=":", lw=1); axi.axhline(TASKB_EXOTICS_P[k], c="grey", ls=":", lw=1)
    axi.set_xticks(x); axi.set_xticklabels(["P_theta"] + LEVELS); axi.set_title(k); axi.grid(alpha=.3)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(os.path.join(DRIVE, "identification.png"), dpi=150); plt.show()

if "base" in results:
    b = results["base"]; retr = [t for t in ("retrain1", "retrain2") if t in results]
    band = {k: max([results[t]["levels"]["C3"]["exotics_C"][k][0] for t in retr+["base"]]) - min([results[t]["levels"]["C3"]["exotics_C"][k][0] for t in retr+["base"]]) for k in EXOTICS} if retr else {}
    print("retrain band at C3 (max-min over base + retrains):", {k: round(v, 4) for k, v in band.items()})
    print(f"\n  {'prior':10s} " + " ".join(f"{k[:8]:>26s}" for k in EXOTICS) + f" {'ESS_C3%':>8s} {'|b|C0':>7s}")
    for tag, s in results.items():
        row = f"  {LAB[tag]:10s} "
        for k in EXOTICS:
            d = s["levels"]["C3"]["exotics_C"][k][0] - b["levels"]["C3"]["exotics_C"][k][0]
            se = np.hypot(s["levels"]["C3"]["exotics_C"][k][1], b["levels"]["C3"]["exotics_C"][k][1]); span = TASKB_EXOTICS_P[k] - TASKB_EXOTICS_Q[k]
            flag = " *" if band and tag not in retr+["base"] and abs(d) > band[k] else "  "
            row += f" {d:+8.4f}({d/se:+5.1f}SE {d/span*100:+4.0f}%){flag}"
        print(row + f" {s['levels']['C3']['ess_C']*100:8.2f} {s['levels']['C0']['beta_raw_norm']:7.2f}")
    mis = [t for t in ("mu25", "kappa6", "rho02") if t in results]
    if len(mis) >= 2:
        sc = {t: float(np.mean([abs(results[t]["levels"]["C3"]["exotics_C"][k][0] - b["levels"]["C3"]["exotics_C"][k][0]) / (TASKB_EXOTICS_P[k]-TASKB_EXOTICS_Q[k]) for k in EXOTICS])) for t in mis}
        obs = sorted(sc, key=sc.get); pred = [t for t in ("mu25", "kappa6", "rho02") if t in sc]
        print("\nmean |spread|/span at C3:", {t: f"{sc[t]*100:.1f}%" for t in obs})
        print("  pre-registered:", " < ".join(pred), " observed:", " < ".join(obs), "->", "HELD" if obs == pred else "FALSIFIED")
        print(f"  corrupted drift mu=0.25: |beta_raw| at C0 = {results['mu25']['levels']['C0']['beta_raw_norm']:.2f} vs base {b['levels']['C0']['beta_raw_norm']:.2f}" if "mu25" in results else "")

Results in `overlay.json` and per prior in `<tag>/sweep/sweep.json`; the verdict goes to `DECISIONS.md` §15.